<a href="https://colab.research.google.com/github/IvanIzmestev/lab6/blob/main/laboratory_work_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
file_path = 'data_variant_9.csv'

data = np.genfromtxt(file_path, delimiter=',', names=True, dtype=None, encoding='utf-8',
                     deletechars='', replace_space='_')

print(f"Файл загружен. Количество строк: {len(data)}")
print(f"Поля данных: {data.dtype.names}")
print("\nПример первых 5 строк:")
for i in range(min(5, len(data))):
    print(data[i])

ValueError: Some errors were detected !
    Line #231989 (got 2 columns instead of 6)

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

file_path = 'data_variant_9.csv'

print("Загрузка файла...")
data = np.genfromtxt(
    file_path,
    delimiter=',',
    names=True,
    dtype=None,
    encoding='utf-8',
    invalid_raise=False,
    loose=True
)

print(f"Файл загружен. Количество строк: {len(data)}")
print(f"Поля данных: {data.dtype.names}")

Загрузка файла...
Файл загружен. Количество строк: 1443534
Поля данных: ('ts', 'line_id', 'dim', 'weight', 'defects', 'machine')


In [ ]:
dtype = np.dtype([
    ('ts', np.int32),
    ('line_id', np.int16),
    ('dim', np.float32),
    ('weight', np.float32),
    ('defects', np.uint8),
    ('machine', np.uint8)
])

structured_data = np.zeros(len(data), dtype=dtype)

structured_data['ts'] = data['ts'].astype(np.int32)
structured_data['line_id'] = data['line_id'].astype(np.int16)
structured_data['dim'] = data['dim'].astype(np.float32)
structured_data['weight'] = data['weight'].astype(np.float32)
structured_data['defects'] = np.clip(data['defects'].astype(np.float32), 0, 255).astype(np.uint8)
structured_data['machine'] = np.clip(data['machine'].astype(np.float32), 0, 255).astype(np.uint8)

print("Структурированный массив создан.")
print(f"Dtype: {structured_data.dtype}")

Структурированный массив создан.
Dtype: [('ts', '<i4'), ('line_id', '<i2'), ('dim', '<f4'), ('weight', '<f4'), ('defects', 'u1'), ('machine', 'u1')]


In [ ]:
num_rows = len(structured_data)
memory_bytes = structured_data.nbytes
memory_mb = memory_bytes / (1024 ** 2)

print(f"Количество строк: {num_rows}")
print(f"Объём памяти: {memory_bytes} байт ({memory_mb:.2f} МБ)")

nan_count = 0
inf_count = 0

for field in ['dim', 'weight', 'defects', 'machine']:
    arr = structured_data[field].astype(np.float64)
    nan_count += np.sum(np.isnan(arr))
    inf_count += np.sum(np.isinf(arr))

total_numeric = num_rows * 4
nan_percent = nan_count / total_numeric * 100
inf_percent = inf_count / total_numeric * 100

print(f"\nКоличество NaN: {nan_count} ({nan_percent:.2f}%)")
print(f"Количество Inf: {inf_count} ({inf_percent:.2f}%)")

if nan_percent > 3:
    print("ПРЕДУПРЕЖДЕНИЕ: Доля NaN превышает 3%!")
if inf_percent > 3:
    print("ПРЕДУПРЕЖДЕНИЕ: Доля Inf превышает 3%!")

Количество строк: 1443534
Объём памяти: 23096544 байт (22.03 МБ)

Количество NaN: 87105 (1.51%)
Количество Inf: 0 (0.00%)


In [ ]:
np.save('step1_initial_data.npy', structured_data)
print("\nИсходный массив сохранён в 'step1_initial_data.npy'")


Исходный массив сохранён в 'step1_initial_data.npy'


In [ ]:
data = structured_data.copy()
tol_min, tol_max = 95, 105

mask_dim_out_of_range = (data['dim'] < tol_min) | (data['dim'] > tol_max)
mask_dim_nan = np.isnan(data['dim'].astype(np.float64))
mask_dim_inf = np.isinf(data['dim'].astype(np.float64))
mask_weight_negative = (data['weight'].astype(np.float64) < 0)
mask_weight_nan = np.isnan(data['weight'].astype(np.float64))
mask_defects_negative = (data['defects'].astype(np.float64) < 0)

mask_anomalies = (mask_dim_out_of_range | mask_dim_nan | mask_dim_inf |
                  mask_weight_negative | mask_weight_nan | mask_defects_negative)

total_rows = len(data)
anomaly_count = np.sum(mask_anomalies)
anomaly_percent = anomaly_count / total_rows * 100

print(f"Строк с аномалиями: {anomaly_count} из {total_rows} ({anomaly_percent:.2f}%)")

print(f"\nДетализация аномалий:")
print(f"  dim вне допусков [{tol_min}, {tol_max}]: {np.sum(mask_dim_out_of_range)} ({np.sum(mask_dim_out_of_range)/total_rows*100:.2f}%)")
print(f"  dim NaN: {np.sum(mask_dim_nan)} ({np.sum(mask_dim_nan)/total_rows*100:.2f}%)")
print(f"  dim Inf: {np.sum(mask_dim_inf)} ({np.sum(mask_dim_inf)/total_rows*100:.2f}%)")
print(f"  weight < 0: {np.sum(mask_weight_negative)} ({np.sum(mask_weight_negative)/total_rows*100:.2f}%)")
print(f"  weight NaN: {np.sum(mask_weight_nan)} ({np.sum(mask_weight_nan)/total_rows*100:.2f}%)")
print(f"  defects < 0: {np.sum(mask_defects_negative)} ({np.sum(mask_defects_negative)/total_rows*100:.2f}%)")

Строк с аномалиями: 1438848 из 1443534 (99.68%)

Детализация аномалий:
  dim вне допусков [95, 105]: 1395321 (96.66%)
  dim NaN: 43247 (3.00%)
  dim Inf: 0 (0.00%)
  weight < 0: 40568 (2.81%)
  weight NaN: 43858 (3.04%)
  defects < 0: 0 (0.00%)


In [ ]:
data_clean = data.copy()

dim_median = np.nanmedian(data['dim'].astype(np.float64))
data_clean['dim'][mask_dim_out_of_range] = dim_median

data_clean['dim'][mask_dim_nan] = dim_median

data_clean['dim'][np.isposinf(data['dim'].astype(np.float64))] = tol_max
data_clean['dim'][np.isneginf(data['dim'].astype(np.float64))] = tol_min

data_clean['weight'][mask_weight_negative] = 0.0

weight_median = np.nanmedian(data['weight'].astype(np.float64))
data_clean['weight'][mask_weight_nan] = weight_median

data_clean['defects'][mask_defects_negative] = 0

q1_weight = np.percentile(data_clean['weight'].astype(np.float64), 25)
q3_weight = np.percentile(data_clean['weight'].astype(np.float64), 75)
iqr_weight = q3_weight - q1_weight
lower_bound_weight = q1_weight - 1.5 * iqr_weight
upper_bound_weight = q3_weight + 1.5 * iqr_weight

print(f"\nIQR для weight: Q1={q1_weight:.2f}, Q3={q3_weight:.2f}, IQR={iqr_weight:.2f}")
print(f"Границы clip: [{lower_bound_weight:.2f}, {upper_bound_weight:.2f}]")

data_clean['weight'] = np.clip(data_clean['weight'].astype(np.float64),
                                lower_bound_weight, upper_bound_weight).astype(np.float32)

print("\nОчистка завершена.")
print("Оставшиеся аномалии:", np.sum((data_clean['dim'] < tol_min) | (data_clean['dim'] > tol_max)))


IQR для weight: Q1=32.59, Q3=67.47, IQR=34.89
Границы clip: [-19.74, 119.80]

Очистка завершена.
Оставшиеся аномалии: 1438568


In [ ]:
line_ids, group_indices, group_counts = np.unique(data_clean['line_id'],
                                                    return_inverse=True,
                                                    return_counts=True)

num_groups = len(line_ids)

print(f"Количество групп (линий): {num_groups}")
print("\nМощность каждой группы (количество строк):")
for i, line_id_val in enumerate(line_ids):
    print(f"  Линия {line_id_val}: {group_counts[i]} строк")

Количество групп (линий): 100

Мощность каждой группы (количество строк):
  Линия 0: 14475 строк
  Линия 1: 14481 строк
  Линия 2: 14624 строк
  Линия 3: 14674 строк
  Линия 4: 14515 строк
  Линия 5: 14425 строк
  Линия 6: 14307 строк
  Линия 7: 14573 строк
  Линия 8: 14460 строк
  Линия 9: 14594 строк
  Линия 10: 14392 строк
  Линия 11: 14391 строк
  Линия 12: 14523 строк
  Линия 13: 14420 строк
  Линия 14: 14539 строк
  Линия 15: 14396 строк
  Линия 16: 14484 строк
  Линия 17: 14365 строк
  Линия 18: 14370 строк
  Линия 19: 14345 строк
  Линия 20: 14478 строк
  Линия 21: 14390 строк
  Линия 22: 14662 строк
  Линия 23: 14280 строк
  Линия 24: 14304 строк
  Линия 25: 14535 строк
  Линия 26: 14331 строк
  Линия 27: 14778 строк
  Линия 28: 14514 строк
  Линия 29: 14402 строк
  Линия 30: 14375 строк
  Линия 31: 14301 строк
  Линия 32: 14398 строк
  Линия 33: 14344 строк
  Линия 34: 14561 строк
  Линия 35: 14409 строк
  Линия 36: 14556 строк
  Линия 37: 14606 строк
  Линия 38: 14383 строк


In [ ]:
mean_dim_by_group = np.zeros(num_groups, dtype=np.float64)
max_defects_by_group = np.zeros(num_groups, dtype=np.uint8)
std_dim_by_group = np.zeros(num_groups, dtype=np.float64)

for i in range(num_groups):
    group_mask = (group_indices == i)
    group_dim = data_clean['dim'][group_mask].astype(np.float64)
    group_defects = data_clean['defects'][group_mask]

    mean_dim_by_group[i] = np.mean(group_dim)
    std_dim_by_group[i] = np.std(group_dim)
    max_defects_by_group[i] = np.max(group_defects) if len(group_defects) > 0 else 0

    print(f"Линия {line_ids[i]}: средний dim = {mean_dim_by_group[i]:.3f} мм, "
          f"макс defects = {max_defects_by_group[i]}, "
          f"среднекв. откл. dim = {std_dim_by_group[i]:.3f} мм")

Линия 0: средний dim = 0.317 мм, макс defects = 255, среднекв. откл. dim = 5.404 мм
Линия 1: средний dim = 0.370 мм, макс defects = 255, среднекв. откл. dim = 5.847 мм
Линия 2: средний dim = 0.354 мм, макс defects = 255, среднекв. откл. dim = 5.714 мм
Линия 3: средний dim = 0.358 мм, макс defects = 255, среднекв. откл. dim = 5.800 мм
Линия 4: средний dim = 0.288 мм, макс defects = 255, среднекв. откл. dim = 5.108 мм
Линия 5: средний dim = 0.378 мм, макс defects = 255, среднекв. откл. dim = 5.910 мм
Линия 6: средний dim = 0.337 мм, макс defects = 255, среднекв. откл. dim = 5.544 мм
Линия 7: средний dim = 0.391 мм, макс defects = 255, среднекв. откл. dim = 6.040 мм
Линия 8: средний dim = 0.379 мм, макс defects = 255, среднекв. откл. dim = 5.926 мм
Линия 9: средний dim = 0.394 мм, макс defects = 255, среднекв. откл. dim = 6.040 мм
Линия 10: средний dim = 0.403 мм, макс defects = 255, среднекв. откл. dim = 6.145 мм
Линия 11: средний dim = 0.387 мм, макс defects = 255, среднекв. откл. dim =

In [ ]:
data_normalized = data_clean.copy()

dtype_normalized = np.dtype(data_normalized.dtype.descr + [('dim_zscore', np.float32)])
data_with_zscore = np.zeros(len(data_normalized), dtype=dtype_normalized)

for field in data_normalized.dtype.names:
    data_with_zscore[field] = data_normalized[field]

dim_zscore = np.zeros(len(data_normalized), dtype=np.float32)

for i in range(num_groups):
    group_mask = (group_indices == i)
    mean_val = mean_dim_by_group[i]
    std_val = std_dim_by_group[i]

    if std_val > 1e-8:
        dim_zscore[group_mask] = (data_normalized['dim'][group_mask].astype(np.float64) - mean_val) / std_val
    else:
        dim_zscore[group_mask] = 0.0

data_with_zscore['dim_zscore'] = dim_zscore

print("Z-score нормализация dim выполнена.")
print(f"Среднее dim_zscore (должно быть ~0): {np.mean(dim_zscore):.6f}")
print(f"Станд. откл. dim_zscore (должно быть ~1): {np.std(dim_zscore):.6f}")

np.save('step3_normalized_data.npy', data_with_zscore)
print("\nНормализованный массив сохранён в 'step3_normalized_data.npy'")

Z-score нормализация dim выполнена.
Среднее dim_zscore (должно быть ~0): -0.000000
Станд. откл. dim_zscore (должно быть ~1): 1.000000

Нормализованный массив сохранён в 'step3_normalized_data.npy'


In [ ]:
k = 25
dim_values = data_clean['dim'].astype(np.float64)
n_rows = len(dim_values)

cumsum = np.cumsum(np.insert(dim_values, 0, 0))
moving_mean = (cumsum[k:] - cumsum[:-k]) / k

cumsum_sq = np.cumsum(np.insert(dim_values ** 2, 0, 0))
moving_mean_sq = (cumsum_sq[k:] - cumsum_sq[:-k]) / k

moving_std = np.sqrt(np.maximum(moving_mean_sq - moving_mean ** 2, 0))

moving_std_full = np.concatenate([np.full(k-1, np.nan), moving_std])

print(f"Скользящее стандартное отклонение dim (k={k}) вычислено.")
print(f"Длина массива: {len(moving_std_full)}")
print(f"Первые 5 значений (NaN для первых {k-1}): {moving_std_full[:k+1]}")
print(f"Среднее moving_std: {np.nanmean(moving_std_full):.4f} мм")

Скользящее стандартное отклонение dim (k=25) вычислено.
Длина массива: 1443534
Первые 5 значений (NaN для первых 24): [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan  0.  0.]
Среднее moving_std: 1.6470 мм


In [ ]:
weight_values = data_clean['weight'].astype(np.float64)
weight_diff = np.diff(weight_values)

weight_diff_full = np.insert(weight_diff, 0, np.nan)

print(f"\nРазница массы (weight_diff) вычислена.")
print(f"Среднее изменение массы: {np.nanmean(weight_diff):.4f} г")
print(f"Максимальное изменение: {np.nanmax(np.abs(weight_diff)):.4f} г")


Разница массы (weight_diff) вычислена.
Среднее изменение массы: 0.0000 г
Максимальное изменение: 119.8024 г


In [ ]:
dtype_extended = np.dtype(data_with_zscore.dtype.descr +
                          [('moving_std_dim', np.float32),
                           ('weight_diff', np.float32)])

data_extended = np.zeros(len(data_with_zscore), dtype=dtype_extended)

for field in data_with_zscore.dtype.names:
    data_extended[field] = data_with_zscore[field]

data_extended['moving_std_dim'] = moving_std_full.astype(np.float32)
data_extended['weight_diff'] = weight_diff_full.astype(np.float32)

print("Новые поля добавлены в массив:")
print(f"  - moving_std_dim (скользящее стандартное отклонение dim)")
print(f"  - weight_diff (изменение массы между соседними замерами)")
print(f"\nИтоговый dtype: {data_extended.dtype.names}")

Новые поля добавлены в массив:
  - moving_std_dim (скользящее стандартное отклонение dim)
  - weight_diff (изменение массы между соседними замерами)

Итоговый dtype: ('ts', 'line_id', 'dim', 'weight', 'defects', 'machine', 'dim_zscore', 'moving_std_dim', 'weight_diff')


In [ ]:
density = np.where(data_extended['dim'].astype(np.float64) > 1e-8,
                   data_extended['weight'].astype(np.float64) / data_extended['dim'].astype(np.float64),
                   0.0)

density_median = np.nanmedian(density)
density = np.where(np.isnan(density) | np.isinf(density), density_median, density)

print(f"Признак 'density' (плотность) создан.")
print(f"  Диапазон: [{np.min(density):.3f}, {np.max(density):.3f}] г/мм")
print(f"  Медиана: {density_median:.3f} г/мм")

Признак 'density' (плотность) создан.
  Диапазон: [0.000, 4593.425] г/мм
  Медиана: 1917.190 г/мм


In [ ]:
defect_rate = np.where(data_extended['weight'].astype(np.float64) > 1e-8,
                       data_extended['defects'].astype(np.float64) / data_extended['weight'].astype(np.float64) * 100,
                       0.0)

defect_rate_median = np.nanmedian(defect_rate)
defect_rate = np.where(np.isnan(defect_rate) | np.isinf(defect_rate), defect_rate_median, defect_rate)

print(f"\nПризнак 'defect_rate' (удельная дефектность) создан.")
print(f"  Диапазон: [{np.min(defect_rate):.4f}, {np.max(defect_rate):.4f}] деф./100г")
print(f"  Медиана: {defect_rate_median:.4f} деф./100г")


Признак 'defect_rate' (удельная дефектность) создан.
  Диапазон: [0.0000, 2077591904.2343] деф./100г
  Медиана: 238.3170 деф./100г


In [ ]:
dtype_final = np.dtype(data_extended.dtype.descr +
                       [('density', np.float32),
                        ('defect_rate', np.float32)])

data_final = np.zeros(len(data_extended), dtype=dtype_final)
for field in data_extended.dtype.names:
    data_final[field] = data_extended[field]

data_final['density'] = density.astype(np.float32)
data_final['defect_rate'] = defect_rate.astype(np.float32)

print("\nФинальный массив с производными признаками создан.")


Финальный массив с производными признаками создан.


In [ ]:
condition_mask = (data_final['defects'] > 0)

print(f"Записей с дефектами: {np.sum(condition_mask)} ({np.sum(condition_mask)/len(data_final)*100:.2f}%)")

Записей с дефектами: 1438000 (99.62%)


In [ ]:
conditional_stats = []

for i in range(num_groups):
    group_mask = (group_indices == i)
    combined_mask = group_mask & condition_mask

    if np.sum(combined_mask) > 0:
        dim_conditional = data_final['dim'][combined_mask].astype(np.float64)
        mean_cond = np.mean(dim_conditional)
        median_cond = np.median(dim_conditional)
    else:
        mean_cond = np.nan
        median_cond = np.nan

    conditional_stats.append((line_ids[i], mean_cond, median_cond))

conditional_array = np.array(conditional_stats, dtype=np.dtype([
    ('line_id', np.int16),
    ('mean_dim_defective', np.float32),
    ('median_dim_defective', np.float32)
]))

print("Условная агрегация (dim для дефектных деталей) по линиям:")
print(f"{'Линия':<8} {'Средний dim':<15} {'Медианный dim'}")
for row in conditional_array:
    print(f"{row['line_id']:<8} {row['mean_dim_defective']:<15.3f} {row['median_dim_defective']:.3f}")

Условная агрегация (dim для дефектных деталей) по линиям:
Линия    Средний dim     Медианный dim
0        0.312           0.026
1        0.364           0.026
2        0.355           0.026
3        0.360           0.026
4        0.289           0.026
5        0.379           0.026
6        0.339           0.026
7        0.386           0.026
8        0.380           0.026
9        0.396           0.026
10       0.404           0.026
11       0.381           0.026
12       0.441           0.026
13       0.338           0.026
14       0.383           0.026
15       0.426           0.026
16       0.301           0.026
17       0.360           0.026
18       0.361           0.026
19       0.376           0.026
20       0.379           0.026
21       0.438           0.026
22       0.417           0.026
23       0.321           0.026
24       0.391           0.026
25       0.288           0.026
26       0.390           0.026
27       0.332           0.026
28       0.414           0.026
29  

In [ ]:
weight_current = data_final['weight'].astype(np.float64)

weight_lag1 = np.roll(weight_current, 1)
weight_lag1[0] = np.nan

weight_change = weight_current - weight_lag1

print(f"Лаговый признак для weight создан.")
print(f"Первые 5 значений weight_change: {weight_change[:5]}")

Лаговый признак для weight создан.
Первые 5 значений weight_change: [         nan  36.21831703  10.91453934 -20.18990707  20.73019791]


In [ ]:
valid_change = weight_change[~np.isnan(weight_change)]

signs = np.sign(valid_change)

bins = np.bincount((signs + 1).astype(int), minlength=3)

decreased = bins[0]
unchanged = bins[1]
increased = bins[2]

total_valid = len(valid_change)

print(f"Распределение изменений weight (всего {total_valid} записей):")
print(f"  Росло (+):     {increased} ({increased/total_valid*100:.2f}%)")
print(f"  Падало (-):    {decreased} ({decreased/total_valid*100:.2f}%)")
print(f"  Без изменений: {unchanged} ({unchanged/total_valid*100:.2f}%)")

unique_signs, counts_signs = np.unique(signs, return_counts=True)
print("\nЧерез np.unique:")
for sign, count in zip(unique_signs, counts_signs):
    sign_name = "Рост (+)" if sign == 1 else "Падение (-)" if sign == -1 else "Без изменений"
    print(f"  {sign_name}: {count} ({count/total_valid*100:.2f}%)")

Распределение изменений weight (всего 1443533 записей):
  Росло (+):     720427 (49.91%)
  Падало (-):    719225 (49.82%)
  Без изменений: 3881 (0.27%)

Через np.unique:
  Падение (-): 719225 (49.82%)
  Без изменений: 3881 (0.27%)
  Рост (+): 720427 (49.91%)


In [ ]:
dtype_lag = np.dtype(data_final.dtype.descr +
                     [('weight_lag1', np.float32),
                      ('weight_change', np.float32)])

data_with_lag = np.zeros(len(data_final), dtype=dtype_lag)
for field in data_final.dtype.names:
    data_with_lag[field] = data_final[field]

data_with_lag['weight_lag1'] = weight_lag1.astype(np.float32)
data_with_lag['weight_change'] = weight_change.astype(np.float32)

print("\nЛаговые поля добавлены в массив.")


Лаговые поля добавлены в массив.


In [ ]:
data_robust = data_with_lag.copy()
outlier_replaced_count = np.zeros(num_groups, dtype=np.int32)

for i in range(num_groups):
    group_mask = (group_indices == i)
    dim_group = data_robust['dim'][group_mask].astype(np.float64)

    sorted_dim = np.sort(dim_group)
    q1 = np.percentile(sorted_dim, 25)
    q3 = np.percentile(sorted_dim, 75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (dim_group < lower_bound) | (dim_group > upper_bound)
    num_outliers = np.sum(outlier_mask)
    outlier_replaced_count[i] = num_outliers

    if num_outliers > 0:
        median_group = np.median(sorted_dim)
        data_robust['dim'][group_mask] = np.where(outlier_mask, median_group, dim_group).astype(np.float32)

    print(f"Линия {line_ids[i]}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
          f"границы=[{lower_bound:.2f}, {upper_bound:.2f}], "
          f"заменено выбросов: {num_outliers}")

Линия 0: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 42
Линия 1: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 50
Линия 2: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 48
Линия 3: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 48
Линия 4: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 38
Линия 5: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 51
Линия 6: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 45
Линия 7: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 53
Линия 8: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 51
Линия 9: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 54
Линия 10: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 54
Линия 11: Q1=0.03, Q3=0.03, IQR=0.00, границы=[0.03, 0.03], заменено выбросов: 52
Линия 12: Q1=0.03, Q3=0.03

In [ ]:
total_replaced = np.sum(outlier_replaced_count)
total_all = len(data_robust)
replaced_percent = total_replaced / total_all * 100

print(f"\nОбщее количество заменённых выбросов: {total_replaced}")
print(f"Общая доля изменённых записей: {replaced_percent:.2f}%")
print(f"По группам: {outlier_replaced_count}")


Общее количество заменённых выбросов: 4966
Общая доля изменённых записей: 0.34%
По группам: [42 50 48 48 38 51 45 53 51 54 54 52 60 45 52 57 40 48 48 50 51 59 57 42
 52 38 52 45 56 42 46 48 56 52 52 40 50 46 41 57 39 44 47 49 41 45 37 47
 49 49 50 48 47 38 54 44 45 57 48 46 40 64 48 61 45 59 67 49 55 53 57 45
 57 49 48 32 55 63 51 64 47 48 60 38 49 42 49 50 50 48 65 43 56 62 49 33
 64 55 54 50]


In [ ]:
tol_min, tol_max = 95, 105

mask_defects_too_high = data_robust['defects'] > 50
mask_weight_non_positive = data_robust['weight'].astype(np.float64) <= 0
mask_dim_out_of_tolerance = (data_robust['dim'].astype(np.float64) < tol_min) | (data_robust['dim'].astype(np.float64) > tol_max)
mask_machine_invalid = (data_robust['machine'] < 1) | (data_robust['machine'] > 10)

mask_violations = (mask_defects_too_high | mask_weight_non_positive |
                   mask_dim_out_of_tolerance | mask_machine_invalid)

num_violations = np.sum(mask_violations)
violation_percent = num_violations / len(data_robust) * 100

print(f"Нарушения логической целостности:")
print(f"  defects > 50: {np.sum(mask_defects_too_high)}")
print(f"  weight <= 0: {np.sum(mask_weight_non_positive)}")
print(f"  dim вне допусков [{tol_min}, {tol_max}]: {np.sum(mask_dim_out_of_tolerance)}")
print(f"  machine вне [1, 10]: {np.sum(mask_machine_invalid)}")
print(f"\n  ВСЕГО нарушений: {num_violations} ({violation_percent:.2f}%)")

data_robust['defects'] = np.where(mask_defects_too_high, 50, data_robust['defects'])
data_robust['weight'] = np.where(mask_weight_non_positive,
                                  np.median(data_robust['weight'].astype(np.float64)),
                                  data_robust['weight'])
data_robust['dim'] = np.where(mask_dim_out_of_tolerance,
                               np.median(data_robust['dim'].astype(np.float64)),
                               data_robust['dim'])
data_robust['machine'] = np.where(mask_machine_invalid, 1, data_robust['machine'])

print("\nНарушения исправлены.")

Нарушения логической целостности:
  defects > 50: 1156423
  weight <= 0: 40568
  dim вне допусков [95, 105]: 1443534
  machine вне [1, 10]: 1155508

  ВСЕГО нарушений: 1443534 (100.00%)

Нарушения исправлены.


In [ ]:
machine_values = data_robust['machine']
unique_machines, machine_counts = np.unique(machine_values, return_counts=True)

total_rows = len(data_robust)
threshold = 0.01 * total_rows

print(f"Распределение категорий machine (порог 1% = {threshold:.0f} записей):")
print(f"{'Станок':<10} {'Количество':<12} {'Доля':<10} {'Статус'}")
print("-" * 45)

rare_machines = []

for machine_val, count in zip(unique_machines, machine_counts):
    percent = count / total_rows * 100
    status = "OK" if count >= threshold else "-> OTHER"
    print(f"{machine_val:<10} {count:<12} {percent:<10.2f}% {status}")
    if count < threshold:
        rare_machines.append(machine_val)

if len(rare_machines) > 0:
    mask_rare = np.isin(data_robust['machine'], rare_machines)
    num_replaced = np.sum(mask_rare)
    data_robust['machine'][mask_rare] = 255

    print(f"\nЗаменено {num_replaced} записей ({num_replaced/total_rows*100:.2f}%) редких станков на OTHER (255)")
    print(f"Редкие станки: {rare_machines}")
else:
    print("\nРедких категорий не найдено.")

Распределение категорий machine (порог 1% = 14435 записей):
Станок     Количество   Доля       Статус
---------------------------------------------
1          1184265      82.04     % OK
2          28735        1.99      % OK
3          28790        1.99      % OK
4          28786        1.99      % OK
5          28835        2.00      % OK
6          28842        2.00      % OK
7          29085        2.01      % OK
8          28615        1.98      % OK
9          28665        1.99      % OK
10         28916        2.00      % OK

Редких категорий не найдено.


In [ ]:
np.save('final_cleaned_data.npy', data_robust)
print("\nФинальный очищенный массив сохранён в 'final_cleaned_data.npy'")
print(f"Итоговый dtype: {data_robust.dtype}")
print(f"Поля: {data_robust.dtype.names}")
print(f"Количество строк: {len(data_robust)}")
print(f"Размер в памяти: {data_robust.nbytes / 1024:.2f} КБ")


Финальный очищенный массив сохранён в 'final_cleaned_data.npy'
Итоговый dtype: [('ts', '<i4'), ('line_id', '<i2'), ('dim', '<f4'), ('weight', '<f4'), ('defects', 'u1'), ('machine', 'u1'), ('dim_zscore', '<f4'), ('moving_std_dim', '<f4'), ('weight_diff', '<f4'), ('density', '<f4'), ('defect_rate', '<f4'), ('weight_lag1', '<f4'), ('weight_change', '<f4')]
Поля: ('ts', 'line_id', 'dim', 'weight', 'defects', 'machine', 'dim_zscore', 'moving_std_dim', 'weight_diff', 'density', 'defect_rate', 'weight_lag1', 'weight_change')
Количество строк: 1443534
Размер в памяти: 62026.85 КБ


In [ ]:
unique_machines = np.unique(machine_values)
print(f"Уникальных станков: {len(unique_machines)}")
print(f"Список уникальных станков: {unique_machines}")
print(f"Есть ли 6 станок? {6 in unique_machines}")

t_machine = 6

mask = ((data_raw['machine'] == t_machine) & ((data_raw['dim'] <= 0) | (data_raw['weight'] >= 100)))
indices = np.where(mask)

print(len(indices[0]))

Уникальных станков: 10
Список уникальных станков: [ 1  2  3  4  5  6  7  8  9 10]
Есть ли 6 станок? True
20411


ИТОГОВАЯ СВОДКА ПО ЛАБОРАТОРНОЙ РАБОТЕ

1. Загружено строк: 1443534
2. Обнаружено аномалий: 1438848 (99.68%)
3. Количество групп (линий): 100
4. Выполнена Z-score нормализация dim по группам
5. Скользящее std dim (k=25): среднее 1.647 мм
6. Создано производных признаков: density, defect_rate
7. Лаговый анализ weight: рост 720427, падение 719225, без изменений 3881
8. Заменено IQR-выбросов: 4966 (0.34%)
9. Исправлено логических нарушений: 1443534 (100.00%)
10. Сжато редких категорий machine: 0 -> OTHER (255)

Лабораторная работа выполнена успешно!


In [ ]:
import numpy as np
